# Label vs axes (design_name, utilization, clock)

In [19]:
import numpy as np
import pandas as pd
import os
from IPython.display import display, Markdown

LABEL_DIR = '/home/spedicato/CircuitNet/routability_ir_drop_prediction/training_set_N28/DRC/label'
THRESHOLD = 0.1


def parse_sample_name(filename: str) -> dict:
    basename = filename.replace('.npy', '')
    parts = basename.split('-')
    if parts[0].isdigit():
        parts = parts[1:]
    for expected, token in zip(['c', 'u', 'm', 'p', 'f'], parts[-5:]):
        if not token.startswith(expected):
            raise ValueError(f'Bad token {token} in {filename}')
    c, u, m, p, f = parts[-5:]

    design_name = '-'.join(parts[:-6])

    m = {
        90: 'high',
        85: 'high',
        80: 'medium',
        75: 'low',
        70: 'low',
    }

    return {
        'design_name':      '-'.join(design_name.split('-')[:-1]),
        'clock_ns':         float(c[1:]),
        'utilization':      m.get(int(float(u[1:]) * 100), 'unknown'),
        'filename':         filename,
    }


files = sorted(f for f in os.listdir(LABEL_DIR) if f.endswith('.npy'))
df_meta = pd.DataFrame([parse_sample_name(f) for f in files])

print(f'Samples: {len(df_meta)}')
for c in ['clock_ns', 'utilization', 'design_name']:
    print(f'  {c:20s} {df_meta[c].nunique():3d} levels')

Samples: 10242
  clock_ns               3 levels
  utilization            3 levels
  design_name            3 levels


## Columns violation rates tables

In [20]:
records = []
for i, row in enumerate(df_meta.itertuples(index=False), 1):
    label = np.load(os.path.join(LABEL_DIR, row.filename))
    label = np.squeeze(label)

    mask = label > THRESHOLD
    records.append({
        'filename':       row.filename,
        'n_tiles':        mask.size,
        'n_violations':   int(mask.sum()),
        'violation_rate': float(mask.mean()),
        'has_violation':  bool(mask.any()),
        'label_max':      float(label.max()),
    })

df = df_meta.merge(pd.DataFrame(records), on='filename')

for c in ['design_name', 'clock_ns', 'utilization']:
    table = (
        df
            .groupby(c)
            .agg(
                n_samples       = ('filename',       'size'),
                p_any_violation = ('has_violation',  'mean'),
                mean_viol_rate  = ('violation_rate', 'mean'),
                std_viol_rate   = ('violation_rate', 'std'),
            )
            .sort_values('mean_viol_rate', ascending=False)
    )

    display(table)

,n_samples,p_any_violation,mean_viol_rate,std_viol_rate
design_name,,,,
RISCY-FPU,3217,0.582530,0.027966,0.067287
RISCY,3861,0.569024,0.011967,0.039297
zero-riscy,3164,0.415929,0.010267,0.039533


,n_samples,p_any_violation,mean_viol_rate,std_viol_rate
clock_ns,,,,
2.0,3250,0.611692,0.027202,0.063928
5.0,4172,0.545062,0.012130,0.042572
20.0,2820,0.398936,0.010509,0.040828


,n_samples,p_any_violation,mean_viol_rate,std_viol_rate
utilization,,,,
high,3495,0.787124,0.037033,0.068526
medium,2216,0.544675,0.016625,0.055986
low,4531,0.315383,0.000525,0.004923


## design_name x utilization_rate x clock violation rates table

In [22]:
keys = ['design_name', 'utilization', 'clock_ns']

combo = (
    df
        .groupby(keys, as_index=False)
        .agg(
            n_samples       = ('filename',       'size'),
            mean_viol_rate  = ('violation_rate', 'mean'),
            std_viol_rate   = ('violation_rate', 'std'),
            p_any_violation = ('has_violation',  'mean')
        )
        .sort_values('mean_viol_rate', ascending=False, ignore_index=True)
)

display(combo)

heat = combo.pivot_table(index=['design_name', 'utilization'],
                         columns='clock_ns',
                         values='mean_viol_rate')
heat = heat.loc[heat.mean(axis=1).sort_values(ascending=False).index]

display(Markdown("### Heatmap of mean violation rate by design, utilization, and clock_ns"))
display(heat)

heat = combo.pivot_table(index=['design_name', 'utilization'],
                         columns='clock_ns',
                         values='p_any_violation')
heat = heat.loc[heat.mean(axis=1).sort_values(ascending=False).index]

display(Markdown("### Heatmap of probability of any violation by design, utilization, and clock_ns"))
display(heat)

,design_name,utilization,clock_ns,n_samples,mean_viol_rate,std_viol_rate,p_any_violation
0,RISCY-FPU,high,2.0,359,0.096642,0.099497,0.974930
1,RISCY-FPU,high,5.0,501,0.054791,0.089176,0.850299
2,RISCY,high,2.0,460,0.044035,0.069130,0.945652
3,RISCY-FPU,high,20.0,234,0.042063,0.072373,0.722222
4,zero-riscy,high,2.0,308,0.039815,0.062768,0.701299
5,RISCY-FPU,medium,2.0,226,0.038671,0.069921,0.831858
6,RISCY,medium,2.0,283,0.028660,0.075286,0.724382
7,zero-riscy,medium,20.0,190,0.026522,0.080734,0.236842
8,RISCY,high,5.0,480,0.019088,0.032498,0.820833
9,zero-riscy,high,20.0,299,0.018443,0.054947,0.535117


### Heatmap of mean violation rate by design, utilization, and clock_ns

,clock_ns,2.0,5.0,20.0
design_name,utilization,,,
RISCY-FPU,high,0.096642,0.054791,0.042063
RISCY,high,0.044035,0.019088,0.013100
zero-riscy,high,0.039815,0.011224,0.018443
RISCY-FPU,medium,0.038671,0.015876,0.014758
zero-riscy,medium,0.014576,0.003507,0.026522
RISCY,medium,0.028660,0.006713,0.006673
RISCY-FPU,low,0.003122,0.000464,0.000129
RISCY,low,0.000226,0.000211,0.000221
zero-riscy,low,0.000287,0.000267,0.000059


### Heatmap of probability of any violation by design, utilization, and clock_ns

,clock_ns,2.0,5.0,20.0
design_name,utilization,,,
RISCY-FPU,high,0.974930,0.850299,0.722222
RISCY,high,0.945652,0.820833,0.699717
zero-riscy,high,0.701299,0.706587,0.535117
RISCY-FPU,medium,0.831858,0.575251,0.423280
RISCY,medium,0.724382,0.574257,0.493878
zero-riscy,medium,0.424731,0.484746,0.236842
RISCY,low,0.376689,0.381729,0.308271
RISCY-FPU,low,0.445946,0.364273,0.210660
zero-riscy,low,0.239796,0.277504,0.145833
